In [1]:
import math
import argparse
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import pytorch_lightning as pl
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# DataModule
# ─────────────────────────────────────────────────────────────────────────────

class GridDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str = ".", batch_size: int = 128):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.batch_size = batch_size
        self.train_ds: List[Data] | None = None
        self.val_ds: List[Data] | None = None

    def prepare_data(self):
        # nothing to download, but Trigger for DDP
        pass

    def setup(self, stage: str | None = None):
        self.train_ds, self.val_ds = torch.load("/home/silvarum/TransPath_Adaptation/gcn_dataset/train_graphs.pt"), torch.load("/home/silvarum/TransPath_Adaptation/gcn_dataset/val_graphs.pt")

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size)

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# LightningModule (GCN)
# ─────────────────────────────────────────────────────────────────────────────

class GCNModule(pl.LightningModule):
    def __init__(self, hidden: int = 64, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.conv1 = GCNConv(5, hidden)
        self.conv2 = GCNConv(hidden, 1)
        self.loss_fn = torch.nn.MSELoss()

    def forward(self, data: Data):
        x, ei, ew = data.x, data.edge_index, data.edge_weight
        x = torch.relu(self.conv1(x, ei, ew))
        x = self.conv2(x, ei, ew).squeeze(-1)
        return x

    def _step(self, batch: Data, stage: str):
        preds = self(batch)
        # mask = batch.valid_mask
        loss = self.loss_fn(preds, batch.y)
        self.log(f"{stage}_mse", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch: Data, batch_idx: int):
        return self._step(batch, "train")

    def validation_step(self, batch: Data, batch_idx: int):
        self._step(batch, "val")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [4]:
dm = GridDataModule(data_dir="/home/silvarum/TransPath_Adaptation/gcn_dataset", batch_size=128)

In [5]:
model = GCNModule(hidden=64, lr=1e-2)

logger = WandbLogger(project="gcn-cf", name="ok data preparation_3", resume="never", reinit=True)

ckpt_cb = ModelCheckpoint(
    monitor="val_mse",          # метрика, за которой следим
    mode="min",                 # «меньше — лучше»
    dirpath="checkpoints/",     # куда класть файлы
    filename="best-{epoch}-{val_mse:.4f}",
    save_top_k=3,               # хранить только лучший
)
trainer = pl.Trainer(
    max_epochs=60,
    logger=logger,
    accelerator="cuda",
    devices=[6],
    callbacks=[ckpt_cb],        # ← добавили
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [6]:
trainer.fit(model, datamodule=dm)

You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


wandb: Currently logged in as: alex26-std (alex26-std-saint-petersburg-state-university). Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


/tmp/ipykernel_1842295/2296665147.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.train_ds, self.val_ds = torch.load("/home/silvarum/TransPath_Adaptation/gcn_datas

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/silvarum/miniconda3/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)
/home/silvarum/miniconda3/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=255` in the `DataLoader` to improve performance.
/home/silvarum/miniconda3/lib/python3.12/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 524288. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/home/silvarum/miniconda3/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the

...

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f32b088b260>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f32cc0d2e40, execution_count=6 error_before_exec=None error_in_exec=name 'exit' is not defined info=<ExecutionInfo object at 7f32cc0d0fe0, raw_cell="trainer.fit(model, datamodule=dm)" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bmkn/home/silvarum/TransPath_Adaptation/gcn.ipynb#X22sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

In [7]:
print("f")

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f32b088b260>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f32221ba810, raw_cell="print("f")" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bmkn/home/silvarum/TransPath_Adaptation/gcn.ipynb#X26sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

f
Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f32b088b260>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f32221bbaa0, execution_count=7 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f32221ba810, raw_cell="print("f")" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bmkn/home/silvarum/TransPath_Adaptation/gcn.ipynb#X26sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe